In [69]:
#!/usr/bin/env julia

using Muspel
using StaticArrays
using HDF5
using Statistics
using Plots

sim_name = "en024048_hion"
snap = 385

atom_cfgs = [
    (
        name = "H",
        atom_file = "/mn/stornext/u3/harshm/Documents/WorkRepo/multi3d/input/atoms/atom.h6_tiago2.yaml",
        pops_file = "/mn/stornext/d9/data/harshm/bifrost_data/$(sim_name)/$(snap)/H/out_pop",
        nlevels = 6,
    ),
]

mesh_file = "/mn/stornext/d9/data/harshm/bifrost_data/$(sim_name)/$(snap)/mesh"
atmos_file = "/mn/stornext/d9/data/harshm/bifrost_data/$(sim_name)/$(snap)/atm3d"
pred_h5 = joinpath(@__DIR__, "training", "original_multi3d_dep_$(sim_name)_$(snap).h5")
pred_key = "departure_coefficients"


function lte_pops_saha(atom, atmos::Atmosphere3D)
    nH = atmos.hydrogen1_density .+ atmos.proton_density
    ratio = 10.0^(atom.abundance - 12.0)
    n_species = ratio .* nH

    pops = Muspel.saha_boltzmann.(
        Ref(atom),
        atmos.temperature,
        atmos.electron_density,
        n_species,
    )

    pops_s = SVector{atom.nlevels,Float32}.(pops)
    reint = reshape(reinterpret(Float32, pops_s), atom.nlevels, size(pops_s)...)
    return permutedims(reint, (2, 3, 4, 1))
end


function load_pred_depcoeff_current(pred_h5::String, pred_key::String)
    h5open(pred_h5, "r") do f
        raw = read(f[pred_key])
        return PermutedDimsArray(raw, (2, 3, 4, 1))
    end
end


function load_pred_depcoeff_alternate(pred_h5::String, pred_key::String)
    h5open(pred_h5, "r") do f
        raw = read(f[pred_key])
        return PermutedDimsArray(raw, (2, 4, 3, 1))
    end
end


function split_atoms(dep_coeff, atoms)
    offsets = cumsum([0; [a.nlevels for a in atoms]])
    out = Dict{String,Any}()

    for (i, a) in enumerate(atoms)
        s = offsets[i] + 1
        e = offsets[i + 1]
        out[a.name] = view(dep_coeff, :, :, :, s:e)
    end

    return out
end


function get_z_scale(atmos)
    if hasproperty(atmos, :z)
        return Float32.(getproperty(atmos, :z))
    elseif hasproperty(atmos, :z_scale)
        return Float32.(getproperty(atmos, :z_scale))
    elseif hasproperty(atmos, :height)
        return Float32.(getproperty(atmos, :height))
    else
        return Float32.(collect(1:atmos.nz))
    end
end


function stats(label, a, b)
    diff = Array(a .- b)
    absdiff = abs.(diff)
    reldiff = ifelse.(b .== 0, absdiff, absdiff ./ abs.(b))

    println(label)
    println("  size(a)      = ", size(a))
    println("  size(b)      = ", size(b))
    println("  max abs diff = ", maximum(absdiff))
    println("  mean abs diff = ", mean(absdiff))
    println("  max rel diff = ", maximum(reldiff))
    println("  mean rel diff = ", mean(reldiff))

    safe_label = replace(strip(lowercase(label)), r"[^a-z0-9]+" => "_")
    hist_path = joinpath(@__DIR__, "$(safe_label)_hist.png")
    weights = fill(1.0 / length(reldiff), length(reldiff))
    p = histogram(
        vec(reldiff),
        bins=1000,
        weights=weights,
        xlabel="rel_diff",
        ylabel="fraction of pixels",
        title=strip(label),
        legend=false,
    )
    savefig(p, hist_path)
    println("  rel diff hist = ", hist_path)
end


println("Reading atmosphere...")
atmos = read_atmos_multi3d(mesh_file, atmos_file)

println("Reading Multi3D pops...")
orig_nlte_atoms = Dict{String,Any}()
orig_lte_atoms = Dict{String,Any}()
orig_dep_atoms = Dict{String,Any}()


for a in atom_cfgs
    atom = Muspel.read_atom(a.atom_file)
    pops_out_nlte, pops_out_lte =
        read_pops_multi3d(a.pops_file, atmos.nx, atmos.ny, atmos.nz, atom.nlevels)

    orig_nlte_atoms[a.name] = pops_out_nlte
    orig_lte_atoms[a.name] = pops_out_lte
    orig_dep_atoms[a.name] = pops_out_nlte ./ pops_out_lte
    println("Atom ", a.name, " direct Multi3D NLTE shape = ", size(pops_out_nlte))
    println("Atom ", a.name, " direct Multi3D LTE shape  = ", size(pops_out_lte))
end

println("Computing LTE pops...")
lte_atoms = Dict{String,Any}()

for a in atom_cfgs
    atom = Muspel.read_atom(a.atom_file)
    pops = lte_pops_saha(atom, atmos)
    lte_atoms[a.name] = pops
    println("Atom ", a.name, " Saha LTE shape          = ", size(pops))
end

dep_coeff_current = load_pred_depcoeff_current(pred_h5, pred_key)
dep_coeff_alternate = load_pred_depcoeff_alternate(pred_h5, pred_key)

println("dep_coeff_current shape = ", size(dep_coeff_current))
println("dep_coeff_alternate shape   = ", size(dep_coeff_alternate))

dep_current_per_atom = split_atoms(dep_coeff_current, atom_cfgs)
dep_alternate_per_atom = split_atoms(dep_coeff_alternate, atom_cfgs)

println("Comparing reconstructed NLTE populations...")

for a in atom_cfgs
    atom = Muspel.read_atom(a.atom_file)

    @assert size(dep_alternate_per_atom[a.name], 4) == atom.nlevels

    nlte_from_current_vs_saha = dep_current_per_atom[a.name] .* lte_atoms[a.name]
    nlte_from_alternate_vs_saha = dep_alternate_per_atom[a.name] .* lte_atoms[a.name]
    nlte_from_alternate_vs_directlte = dep_alternate_per_atom[a.name] .* orig_lte_atoms[a.name]

    println()
    println("Atom ", a.name)
    stats("  current loader vs direct Multi3D NLTE", nlte_from_current_vs_saha, orig_nlte_atoms[a.name])
    stats("  alternate loader with Saha LTE vs direct Multi3D NLTE", nlte_from_alternate_vs_saha, orig_nlte_atoms[a.name])
    stats("  alternate loader with direct Multi3D LTE vs direct Multi3D NLTE", nlte_from_alternate_vs_directlte, orig_nlte_atoms[a.name])
    stats("  Saha LTE vs direct Multi3D LTE", lte_atoms[a.name], orig_lte_atoms[a.name])
    stats("  Dep Current HDF5 vs DEP Multi3D", dep_current_per_atom[a.name], orig_dep_atoms[a.name])
    stats("  Dep New HDF5 vs DEP Multi3D", dep_alternate_per_atom[a.name], orig_dep_atoms[a.name])
end


Reading atmosphere...
Reading Multi3D pops...


[ Info: Could not parse Zeeman components for transition H BF N=2 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=3 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=3 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=3
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=3
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=4


Atom H direct Multi3D NLTE shape = (467, 252, 252, 6)
Atom H direct Multi3D LTE shape  = (467, 252, 252, 6)
Computing LTE pops...
Atom H

[ Info: Could not parse Zeeman components for transition H BF N=2 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=3 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=3 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=3
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=3
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=4


 Saha LTE shape          = (467, 252, 252, 6)
dep_coeff_current shape = (467, 252, 252, 6)
dep_coeff_alternate shape   = (467, 252, 252, 6)
Comparing reconstructed NLTE populations...

Atom 

[ Info: Could not parse Zeeman components for transition H BF N=2 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=3 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=1
[ Info: Could not parse Zeeman components for transition H BF N=3 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=2
[ Info: Could not parse Zeeman components for transition H BF N=4 → H BF N=3
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=3
[ Info: Could not parse Zeeman components for transition H BF N=5 → H BF N=4


H
  current loader vs direct Multi3D NLTE
  size(a)      = (467, 252, 252, 6)
  size(b)      = (467, 252, 252, 6)
  max abs diff = 3.4797585e22
  mean abs diff = 2.7004454e20
  max rel diff = 0.19144262
  mean rel diff = 0.0022058624
  rel diff hist = /mn/stornext/d9/data/harshm/FFNOML/current_loader_vs_direct_multi3d_nlte_hist.png
  alternate loader with Saha LTE vs direct Multi3D NLTE
  size(a)      = (467, 252, 252, 6)
  size(b)      = (467, 252, 252, 6)
  max abs diff = 1.6231197e35
  mean abs diff = 1.6415254e29
  max rel diff = 4.5031442e18
  mean rel diff = 2.6170302e12
  rel diff hist = /mn/stornext/d9/data/harshm/FFNOML/alternate_loader_with_saha_lte_vs_direct_multi3d_nlte_hist.png
  alternate loader with direct Multi3D LTE vs direct Multi3D NLTE
  size(a)      = (467, 252, 252, 6)
  size(b)      = (467, 252, 252, 6)
  max abs diff = 1.6231187e35
  mean abs diff = 1.6415237e29
  max rel diff = 4.503145e18
  mean rel diff = 2.617012e12
  rel diff hist = /mn/stornext/d9/data/har